In [ ]:
!pip install numpy pandas scikit-learn

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('breast-cancer.csv')
df.head()

In [10]:
df.drop(columns=['id'], inplace=True)

In [ ]:
df.head()

In [ ]:
df.shape

In [13]:
X = df.iloc[:, 1:]
y = df.iloc[:, 0]

encoder = LabelEncoder()
y = encoder.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
)

In [ ]:
df.dtypes


In [15]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
x_train

In [ ]:
y_train

In [18]:
y_train.shape

(455,)

In [19]:
import torch

In [20]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [21]:
x_train_tensor.shape

torch.Size([455, 30])

In [ ]:
x_train_tensor

In [41]:
class MYsimpleNN():
    def __init__(self,x):

        self.weights = torch.rand(x.shape[1], 1, dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype = torch.float64,requires_grad=True)
    def forward(self,x):
        y_pred = torch.sigmoid(torch.matmul(x,self.weights) + self.bias)
        return y_pred
    def binarycrossentropyloss(self,y_pred,y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        loss = -torch.mean(y_train_tensor*torch.log(y_pred) + (1-y_train_tensor)*torch.log(1-y_pred))
        return loss

In [74]:
learning_rate = 0.1
epochs = 50

In [75]:
model = MYsimpleNN(x_train_tensor)
model.weights

tensor([[0.2405],
        [0.2233],
        [0.8691],
        [0.1731],
        [0.1896],
        [0.3290],
        [0.2251],
        [0.8359],
        [0.5388],
        [0.5454],
        [0.2694],
        [0.4681],
        [0.7274],
        [0.7391],
        [0.0282],
        [0.5881],
        [0.1261],
        [0.1481],
        [0.8291],
        [0.4350],
        [0.8593],
        [0.6497],
        [0.0349],
        [0.6156],
        [0.7357],
        [0.4841],
        [0.4670],
        [0.7801],
        [0.0716],
        [0.4382]], dtype=torch.float64, requires_grad=True)

In [76]:
model.bias

tensor([0.], dtype=torch.float64, requires_grad=True)

In [92]:
# creating model
model = MYsimpleNN(x_train_tensor)

# define loop
for epoch in range(epochs):
    # forward pass
    y_pred = model.forward(x_train_tensor)
    # print(y_pred)
    # loss calculation
    loss = model.binarycrossentropyloss(y_pred,y_train_tensor)
    # back propagation
    loss.backward()
    # update weights and bias  
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

Epoch 1/50, Loss: 3.2103701288033055
Epoch 2/50, Loss: 3.067004192509942
Epoch 3/50, Loss: 2.9160002344487164
Epoch 4/50, Loss: 2.758301777709823
Epoch 5/50, Loss: 2.593026376179524
Epoch 6/50, Loss: 2.4216939572371463
Epoch 7/50, Loss: 2.2476861836621396
Epoch 8/50, Loss: 2.0755559127743495
Epoch 9/50, Loss: 1.9039857745088236
Epoch 10/50, Loss: 1.7292010877921207
Epoch 11/50, Loss: 1.5605794874000802
Epoch 12/50, Loss: 1.4035785020153624
Epoch 13/50, Loss: 1.2606623222553732
Epoch 14/50, Loss: 1.1344433638639984
Epoch 15/50, Loss: 1.0272773585022532
Epoch 16/50, Loss: 0.9406433010932671
Epoch 17/50, Loss: 0.8745572516131351
Epoch 18/50, Loss: 0.8272131383651032
Epoch 19/50, Loss: 0.795102543690485
Epoch 20/50, Loss: 0.773925470530114
Epoch 21/50, Loss: 0.7598148652118837
Epoch 22/50, Loss: 0.749986546046389
Epoch 23/50, Loss: 0.7427006259267628
Epoch 24/50, Loss: 0.7369512019365909
Epoch 25/50, Loss: 0.7321750038133694
Epoch 26/50, Loss: 0.7280562820269926
Epoch 27/50, Loss: 0.724413

In [93]:
model.weights

tensor([[ 0.5559],
        [ 0.2910],
        [-0.0570],
        [-0.1264],
        [ 0.1239],
        [ 0.1971],
        [-0.3134],
        [-0.4138],
        [-0.0478],
        [ 0.4496],
        [ 0.1981],
        [ 0.0182],
        [-0.2656],
        [-0.1402],
        [-0.1625],
        [ 0.0397],
        [-0.1816],
        [ 0.2179],
        [ 0.2186],
        [-0.1806],
        [ 0.1998],
        [-0.1981],
        [-0.0348],
        [ 0.3558],
        [ 0.2586],
        [-0.2703],
        [ 0.2019],
        [-0.3648],
        [-0.0772],
        [ 0.0346]], dtype=torch.float64, requires_grad=True)

In [94]:
model.bias

tensor([-0.3371], dtype=torch.float64, requires_grad=True)

In [98]:
with torch.no_grad():
    y_test_pred = model.forward(x_test_tensor)
    y_test_pred_labels = (y_test_pred > 0.5).float()
    accuracy = (y_test_pred_labels.squeeze() == y_test_tensor).float().mean()
    print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

Test Accuracy: 58.77%
